# 1.5 Training

(1) Load featured train, (2) tune and compare models with cross-validation and keep the best, (3) save pickle and config.

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_featured_train.csv` |
| Output Model | `1-experimentation/models/model.pkl` |
| Output Config | `1-experimentation/models/model_config.json` |


In [1]:
%pip install -q pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import pickle
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold

TARGET = "price"
RANDOM_STATE = 42
CV_SPLITS = 5

FEATURED_TRAIN_PATH = "../data/data_featured_train.csv"
MODEL_PATH = "../models/model.pkl"
MODEL_CONFIG_PATH = "../models/model_config.json"

Path("../models").mkdir(parents=True, exist_ok=True)


## 1. Load featured train

In [3]:
train_df = pd.read_csv(FEATURED_TRAIN_PATH)

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

print(f"Rows: {len(X_train)} | Features: {X_train.shape[1]}")
print(f"Columns: {list(X_train.columns)}")
X_train.head()


Rows: 67 | Features: 11
Columns: ['sqft', 'bedrooms', 'bathrooms', 'house_age', 'condition', 'location_Downtown', 'location_Mountain', 'location_Rural', 'location_Suburb', 'location_Urban', 'location_Waterfront']


,sqft,bedrooms,bathrooms,house_age,condition,location_Downtown,location_Mountain,location_Rural,location_Suburb,location_Urban,location_Waterfront
0,-0.346309,0.226617,-0.201016,0.025525,2.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.741019,-1.038661,-0.813201,1.062004,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.095767,0.226617,-0.201016,-0.233594,2.0,1.0,0.0,0.0,0.0,0.0,0.0
3,-0.772596,-1.038661,-0.201016,0.699236,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.930481,-1.038661,-0.813201,1.113828,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Tune, compare models and fit the best

Each candidate is tuned with `GridSearchCV` (5-fold CV on train). The search with the lowest RMSE wins; `refit=True` fits that estimator on all of train.


In [4]:
candidates = {
    "linear_regression": (LinearRegression(), {}),
    "ridge": (
        Ridge(),
        {"alpha": [0.1, 1.0, 10.0, 100.0]},
    ),
    "random_forest": (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100],
            "max_depth": [3, 5, None],
            "min_samples_leaf": [1, 2],
        },
    ),
    "gradient_boosting": (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100],
            "learning_rate": [0.05, 0.1],
            "max_depth": [2, 3],
        },
    ),
}

cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}

BEST_SELECTION_METRIC = "mae"

searches = {}
rows = []
for name, (estimator, param_grid) in candidates.items():
    search = GridSearchCV(
        estimator,
        param_grid,
        cv=cv,
        scoring=scoring,
        refit=BEST_SELECTION_METRIC,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best_idx = search.best_index_
    rows.append(
        {
            "model": name,
            "cv_r2": search.cv_results_["mean_test_r2"][best_idx],
            "cv_mae": -search.cv_results_["mean_test_mae"][best_idx],
            "cv_rmse": -search.cv_results_["mean_test_rmse"][best_idx],
            "best_params": search.best_params_,
        }
    )

comparison = pd.DataFrame(rows).sort_values("cv_"+BEST_SELECTION_METRIC).reset_index(drop=True)
display(comparison)

best_name = comparison.loc[0, "model"]
model = searches[best_name].best_estimator_

print(f"Best model: {best_name}")
print(f"Best params: {comparison.loc[0, 'best_params']}")
model


,model,cv_r2,cv_mae,cv_rmse,best_params
0,random_forest,0.984864,21857.421612,39379.794041,"{'max_depth': None, 'min_samples_leaf': 1, 'n_..."
1,gradient_boosting,0.986983,22148.014004,35069.454010,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti..."
2,ridge,0.984209,25478.742367,36160.163507,{'alpha': 0.1}
3,linear_regression,0.982742,26417.987627,37167.718332,{}


Best model: random_forest
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}


RandomForestRegressor(random_state=42)

## 3. Save artifacts

In [5]:
best_metrics = comparison.loc[0, ["cv_r2", "cv_mae", "cv_rmse"]].astype(float).to_dict()

model_config = {
    "model_name": best_name,
    "model_class": type(model).__name__,
    "params": model.get_params(),
    "best_params": comparison.loc[0, "best_params"],
    "features": list(X_train.columns),
    "target": TARGET,
    "cv": {
        "n_splits": CV_SPLITS,
        "random_state": RANDOM_STATE,
        "selection_metric": "cv_"+BEST_SELECTION_METRIC,
        "metrics": best_metrics,
    },
}

with open(MODEL_PATH, "wb") as file:
    pickle.dump(model, file)

with open(MODEL_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=2, default=str)

print(f"Wrote {MODEL_PATH} ({best_name})")
print(f"Wrote {MODEL_CONFIG_PATH}")


Wrote ../models/model.pkl (random_forest)
Wrote ../models/model_config.json
